# webSLM — build a domain SLM for the browser, in Colab

Compiles a domain-specific Small Language Model to a WebGPU `.wasm` + quantized weights
(TVM + mlc-llm from source, pinned to `v0.19.0` — the last release before the `apache-tvm-ffi`
migration that breaks the current wheels). **Ordinary chat SLM — no latent loop, no model
patching.** Model-agnostic: Qwen2/2.5, Llama, Gemma2, Phi-3/3.5, Mistral, ...

**Before you start:** Runtime → Change runtime type → **T4 GPU** (free). The GPU isn't needed
for the WebGPU compile (that's emcc codegen) but speeds up the optional LoRA merge. The TVM
build is ~25-35 min and stays built for the whole session, so you can re-run the compile cell
freely while trying different models.

## 1. Check the runtime

In [ ]:
!nvidia-smi || echo 'no GPU (ok for compile; speeds up the optional LoRA merge)'
!python --version

## 2. System build deps (LLVM + Polly + ninja + cmake)

In [ ]:
import subprocess, glob, os
os.environ['DEBIAN_FRONTEND'] = 'noninteractive'

print('🔄 Clearing apt locks (Colab unattended-upgrades holds them and apt hangs)...')
subprocess.run('pkill -9 -f unattended-upgrade 2>/dev/null || true', shell=True)
subprocess.run('rm -f /var/lib/dpkg/lock-frontend /var/lib/dpkg/lock '
               '/var/cache/apt/archives/lock /var/lib/apt/lists/lock; '
               'dpkg --configure -a 2>/dev/null || true', shell=True)

print('🔄 Updating packages...')
!apt-get update -qq

print('🔄 Installing build dependencies...')
!apt-get install -y -o Dpkg::Use-Pty=0 \
    cmake ninja-build build-essential git \
    llvm-dev libedit-dev libxml2-dev zlib1g-dev libssl-dev

lv = subprocess.check_output(['llvm-config', '--version']).decode().split('.')[0].strip()
print(f'✅ LLVM major: {lv}')
!apt-get install -y -o Dpkg::Use-Pty=0 libpolly-{lv}-dev || apt-get install -y libpolly-18-dev || true
print('✅ libPolly.a:', (glob.glob('/usr/lib/**/libPolly.a', recursive=True) or ['MISSING'])[0])

## 3. Rust (for tokenizers) — force the reliable sparse crates.io index

In [ ]:
import os, shutil
os.environ['PATH'] = '/root/.cargo/bin:' + os.environ['PATH']
if not shutil.which('rustup'):
    !curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs -o /tmp/rustup-init.sh
    !sh /tmp/rustup-init.sh -y --profile minimal --no-modify-path
!rustup default stable
os.environ['CARGO_REGISTRIES_CRATES_IO_PROTOCOL'] = 'sparse'
os.environ['CARGO_NET_GIT_FETCH_WITH_CLI'] = 'true'
!cargo --version

## 4. Emscripten (for the WebGPU `.wasm` target)

In [ ]:
%cd /content
!git clone https://github.com/emscripten-core/emsdk.git
!cd emsdk && ./emsdk install 3.1.56 && ./emsdk activate 3.1.56
import subprocess, os
envtxt = subprocess.check_output('source /content/emsdk/emsdk_env.sh >/dev/null 2>&1 && env',
                                 shell=True, executable='/bin/bash').decode()
for line in envtxt.splitlines():
    if '=' in line:
        k, v = line.split('=', 1)
        if k == 'PATH' or k.startswith('EM') or k == 'EMSDK':
            os.environ[k] = v
!emcc --version | head -1

## 5. Clone mlc-llm `v0.19.0` (+ this repo for merge_lora.py)

In [ ]:
%cd /content
!git clone --branch v0.19.0 --depth 1 --recursive --shallow-submodules https://github.com/mlc-ai/mlc-llm.git
!git clone --depth 1 https://github.com/yourname/webSLM.git

## 6. Build TVM from source  (~25-35 min, once per session)

In [ ]:
%%bash
set -e
cd /content/mlc-llm/3rdparty/tvm
mkdir -p build && cp cmake/config.cmake build/config.cmake
{
  echo 'set(CMAKE_BUILD_TYPE RelWithDebInfo)'
  echo 'set(USE_LLVM "llvm-config --link-static")'
  echo 'set(HIDE_PRIVATE_SYMBOLS ON)'
  echo 'set(USE_CUDA OFF)'; echo 'set(USE_METAL OFF)'
  echo 'set(USE_VULKAN OFF)'; echo 'set(USE_OPENCL OFF)'
} >> build/config.cmake
cd build && cmake .. -G Ninja && ninja

In [ ]:
import os
os.environ['TVM_HOME'] = '/content/mlc-llm/3rdparty/tvm'
os.environ['TVM_LIBRARY_PATH'] = '/content/mlc-llm/3rdparty/tvm/build'
os.environ['LD_LIBRARY_PATH'] = '/content/mlc-llm/3rdparty/tvm/build:' + os.environ.get('LD_LIBRARY_PATH', '')
!pip install -q -e /content/mlc-llm/3rdparty/tvm/python
!python -c "import tvm; print('TVM OK', tvm.__file__)"

## 7. Build mlc-llm  (~3-5 min)

In [ ]:
%%bash
set -e
cd /content/mlc-llm
sed -i 's|set(TOKENIZERS_CPP_RUST_FLAGS "")|set(TOKENIZERS_CPP_RUST_FLAGS "--cap-lints=allow")|' 3rdparty/tokenizers-cpp/CMakeLists.txt
mkdir -p build
{
  echo 'set(CMAKE_BUILD_TYPE RelWithDebInfo)'
  echo 'set(USE_LLVM "llvm-config --link-static")'
  echo 'set(HIDE_PRIVATE_SYMBOLS ON)'
  echo 'set(TVM_SOURCE_DIR /content/mlc-llm/3rdparty/tvm)'
  echo 'set(USE_CUDA OFF)'; echo 'set(USE_METAL OFF)'
  echo 'set(USE_VULKAN OFF)'; echo 'set(USE_OPENCL OFF)'
} > build/config.cmake
cd build && cmake .. -G Ninja && ninja

In [ ]:
!pip install -q -e /content/mlc-llm/python
!pip install -q -U huggingface_hub
!python -c "import tvm, mlc_llm; print('mlc_llm OK', mlc_llm.__file__)"

## 8. (Optional) Merge a LoRA adapter into the base
Skip this unless you fine-tuned with LoRA. Produces a merged checkpoint to compile below;
set `MODEL` to the merged dir in the next cell.

In [ ]:
# !pip install -q peft transformers
# !python /content/webSLM/merge_lora.py \
#     --base Qwen/Qwen2.5-0.5B-Instruct \
#     --adapter /content/my-domain-lora \
#     --out /content/merged/WebSLM-Medical-0.5B

## 9. Convert weights + compile to WebGPU `.wasm`
Set the model variables, then run. Re-run freely while iterating — TVM/mlc-llm stay built.
Examples: Code → `Qwen/Qwen2.5-Coder-1.5B-Instruct`; Math → `Qwen/Qwen2.5-Math-1.5B-Instruct`;
a fine-tune → a local merged dir from step 8.

In [ ]:
%%bash
set -e
# ---- EDIT THESE ----
MODEL=Qwen/Qwen2.5-Coder-1.5B-Instruct   # HF id OR a local merged checkpoint dir
ARCH=qwen2                                # qwen2 | llama | gemma2 | phi3 | mistral
CONV=qwen2                                # qwen2 | llama-3 | gemma_instruction | phi-3 | mistral_default
NAME=WebSLM-Code-1.5B
QUANT=q4f16_1                             # q4f32_1 if you hit NaNs (f16 overflow)
# --------------------
export MLC_LLM_SOURCE_DIR=/content/mlc-llm
TVM=/content/mlc-llm/3rdparty/tvm
(cd /content/mlc-llm/web && TVM_SOURCE_DIR=$TVM make dist/wasm/mlc_wasm_runtime.bc)
(cd $TVM/web && make dist/wasm/wasm_runtime.bc dist/wasm/tvmjs_support.bc dist/wasm/webgpu_runtime.bc)
cp $TVM/web/dist/wasm/*.bc $TVM/build/
cd /content
if [ -d "$MODEL" ]; then SRC="$MODEL"; else SRC=hf/$NAME; hf download $MODEL --local-dir $SRC; fi
W=dist-model/$NAME-MLC; L=dist-model/libs; mkdir -p $W $L
mlc_llm convert_weight $SRC --quantization $QUANT --model-type $ARCH -o $W
mlc_llm gen_config $SRC --quantization $QUANT --model-type $ARCH \
    --conv-template $CONV --prefill-chunk-size 1024 -o $W
mlc_llm compile $W/mlc-chat-config.json --device webgpu \
    -o $L/$NAME-$QUANT-webgpu.wasm
ls -la $L $W

## 10. Upload to Hugging Face
Get a **Write** token at https://huggingface.co/settings/tokens, then run the login cell and
paste it. The repo is created automatically on first upload.

In [ ]:
from huggingface_hub import login
login()  # paste your HF write token

In [ ]:
# Push weights AND the wasm so the HF repo is self-contained.
!hf upload yourname/WebSLM-Code-1.5B-MLC dist-model/WebSLM-Code-1.5B-MLC .
!hf upload yourname/WebSLM-Code-1.5B-MLC dist-model/libs/WebSLM-Code-1.5B-q4f16_1-webgpu.wasm libs/WebSLM-Code-1.5B-q4f16_1-webgpu.wasm

---
**Notes**
- Same recipe as `.github/workflows/build-slm.yml`, just interactive.
- Load the `.wasm` with `@mlc-ai/web-llm` **0.2.79** (matches mlc-llm v0.19.0) — pin both.
- You can't test WebGPU in Colab; load the model in `demo/index.html` or your WebLLM app.